# 🧠 Autograd y Grafos de Computación — Guía Educativa

**Universidad Privada del Norte — Escuela de Posgrado**  
**Bloque B: Ingeniería y Frameworks**  
**Tema: Tinygrad — Entendiendo el autograd y grafos de computación desde cero**

---

## ¿Qué vamos a aprender?

En este notebook aprenderemos **paso a paso** cómo las redes neuronales "aprenden". No necesitas saber matemáticas avanzadas — empezaremos desde lo más básico.

| Sección | Tema | Dificultad |
|---------|------|------------|
| 1 | ¿Qué es una derivada? (con ejemplos numéricos) | ⭐ |
| 2 | La Regla de la Cadena (paso a paso) | ⭐⭐ |
| 3 | Grafos de Computación (dibujar operaciones) | ⭐⭐ |
| 4 | Construyendo nuestro motor autograd | ⭐⭐⭐ |
| 5 | Una neurona que aprende | ⭐⭐⭐ |
| 6 | Red neuronal completa (MLP) | ⭐⭐⭐⭐ |
| 7 | Comparación con tinygrad real | ⭐⭐ |

---
# Sección 1: ¿Qué es una Derivada?

## La analogía del velocímetro 🚗

Imagina que vas en un carro:
- La **posición** del carro es como el valor de una función: `f(x)`
- La **velocidad** del carro es la derivada: `f'(x)`
- La velocidad te dice **qué tan rápido cambia** la posición

### En términos simples:
> **La derivada nos dice: si cambio un poquito la entrada, ¿cuánto cambia la salida?**

### Ejemplo concreto:
Si `f(x) = x²`, y estamos en `x = 3`:
- `f(3) = 9`
- `f(3.001) = 9.006001`
- Cambio en salida ÷ cambio en entrada = `0.006001 / 0.001 ≈ 6.0`
- La derivada de `x²` es `2x`, y `2 × 3 = 6` ✅

¡Veamos esto en código!

In [ ]:
# =============================================================
# EJEMPLO 1: Calculando derivadas "a mano" (método numérico)
# =============================================================

def f(x):
    """Nuestra función: f(x) = x²"""
    return x ** 2

# Punto donde queremos la derivada
x = 3.0
h = 0.001  # Un cambio muy pequeñito

# Cálculo numérico de la derivada
derivada_numerica = (f(x + h) - f(x)) / h

# Derivada exacta (sabemos que d/dx de x² = 2x)
derivada_exacta = 2 * x

print("="*50)
print("CALCULANDO LA DERIVADA DE f(x) = x²")
print("="*50)
print(f"")
print(f"Punto: x = {x}")
print(f"f({x}) = {f(x)}")
print(f"f({x + h}) = {f(x + h)}")
print(f"")
print(f"Derivada numérica:  {derivada_numerica:.4f}")
print(f"Derivada exacta:    {derivada_exacta:.4f}")
print(f"")
print(f"¡Son prácticamente iguales! ✓")

In [ ]:
# =============================================================
# EJEMPLO 2: Visualizando la derivada como pendiente
# =============================================================
import matplotlib.pyplot as plt
import numpy as np

x_vals = np.linspace(0, 5, 100)
y_vals = x_vals ** 2

# Punto de interés
x0 = 3.0
y0 = x0 ** 2
pendiente = 2 * x0  # La derivada en x=3

# Línea tangente: y = pendiente * (x - x0) + y0
tangente_y = pendiente * (x_vals - x0) + y0

plt.figure(figsize=(8, 5))
plt.plot(x_vals, y_vals, 'b-', linewidth=2, label='f(x) = x²')
plt.plot(x_vals, tangente_y, 'r--', linewidth=2, label=f'Tangente (pendiente = {pendiente})')
plt.plot(x0, y0, 'ro', markersize=10, label=f'Punto ({x0}, {y0})')
plt.ylim(-2, 30)
plt.xlim(0, 5)
plt.xlabel('x', fontsize=12)
plt.ylabel('f(x)', fontsize=12)
plt.title('La derivada es la pendiente de la línea tangente', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("La línea roja toca la curva en un solo punto.")
print(f"Su pendiente ({pendiente}) es la derivada de f(x)=x² en x={x0}.")

### 💡 ¿Por qué nos importan las derivadas?

Porque las redes neuronales **aprenden ajustando sus pesos**, y la derivada les dice:

> "Si subo este peso un poquito, ¿el error sube o baja?"

- Si la derivada es **positiva** → subir el peso **aumenta** el error → hay que **bajar** el peso
- Si la derivada es **negativa** → subir el peso **reduce** el error → hay que **subir** el peso

Esto es exactamente lo que hace el **descenso de gradiente**.

---
# Sección 2: La Regla de la Cadena

## La analogía de las fichas de dominó 🁃

Imagina una cadena de fichas de dominó:
- Empujas la primera ficha un poco → la segunda se mueve → la tercera se mueve
- El efecto se **multiplica** en cada eslabón

En matemáticas, si tenemos:
```
a = 2          (entrada)
b = a × 3      (primera operación)
c = b + 1      (segunda operación)
```

Para saber cómo `a` afecta a `c`, multiplicamos los efectos:
- ¿Cómo `a` afecta a `b`? → `db/da = 3` (porque `b = a×3`)
- ¿Cómo `b` afecta a `c`? → `dc/db = 1` (porque `c = b+1`)
- ¿Cómo `a` afecta a `c`? → `dc/da = dc/db × db/da = 1 × 3 = 3`

**¡Se multiplican los efectos!** Esto es la regla de la cadena.

In [ ]:
# =============================================================
# EJEMPLO 3: Regla de la cadena con números
# =============================================================

print("REGLA DE LA CADENA — Ejemplo paso a paso")
print("="*50)
print()

# Definimos una cadena de operaciones
a = 2.0
b = a * 3        # b = 6
c = b + 1        # c = 7
d = c ** 2       # d = 49

print(f"Paso 1: a = {a}")
print(f"Paso 2: b = a × 3 = {b}")
print(f"Paso 3: c = b + 1 = {c}")
print(f"Paso 4: d = c² = {d}")
print()

# Derivadas locales (cada eslabón)
dd_dc = 2 * c     # d(c²)/dc = 2c = 14
dc_db = 1          # d(b+1)/db = 1
db_da = 3          # d(a*3)/da = 3

print("Derivadas locales (cada eslabón):")
print(f"  dd/dc = 2×c = 2×{c} = {dd_dc}")
print(f"  dc/db = {dc_db}")
print(f"  db/da = {db_da}")
print()

# Regla de la cadena: multiplicar todos los eslabones
dd_da = dd_dc * dc_db * db_da
print(f"Regla de la cadena: dd/da = {dd_dc} × {dc_db} × {db_da} = {dd_da}")
print()

# Verificación numérica
h = 0.0001
a2 = a + h
b2 = a2 * 3
c2 = b2 + 1
d2 = c2 ** 2
derivada_numerica = (d2 - d) / h

print(f"Verificación numérica: {derivada_numerica:.2f}")
print(f"¡Coincide! ✓")

### 🔑 Resumen de la Regla de la Cadena

```
Si:  a → b → c → d

Entonces:  dd/da = dd/dc × dc/db × db/da
```

Es como preguntar en una cadena de producción:
- Si cambio la materia prima un 1%, ¿cuánto cambia el producto final?
- Multiplico el efecto de cada etapa.

---
# Sección 3: Grafos de Computación

## ¿Qué es un grafo de computación?

Es simplemente un **diagrama que muestra las operaciones** que hicimos con nuestros datos.

### Analogía: Receta de cocina 👨‍🍳

Imagina que estás haciendo una torta:
```
harina ──┐
          ├── [mezclar] ── masa ──┐
huevos ──┘                        ├── [hornear] ── torta
                     azúcar ──────┘
```

Cada ingrediente es un **nodo**, cada operación es una **conexión**.

En matemáticas, la expresión `y = (a + b) × c` se ve así:
```
a ──┐
     ├── [+] ── temp ──┐
b ──┘                   ├── [×] ── y
               c ───────┘
```

Vamos a dibujar estos grafos con código.

In [ ]:
# =============================================================
# EJEMPLO 4: Dibujando un grafo de computación
# =============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# Nodos (variables)
nodos = {
    'a=2':  (0.5, 3.5),
    'b=3':  (0.5, 1.5),
    '+':    (3.0, 2.5),
    't=5':  (5.0, 2.5),
    'c=4':  (3.0, 0.5),
    '×':    (7.0, 1.5),
    'y=20': (9.0, 1.5),
}

# Dibujar nodos
for nombre, (x, y) in nodos.items():
    if nombre in ['+', '×']:
        circle = plt.Circle((x, y), 0.35, color='#FF6B6B', ec='black', lw=2)
        ax.add_patch(circle)
        ax.text(x, y, nombre, ha='center', va='center', fontsize=16, fontweight='bold', color='white')
    else:
        rect = patches.FancyBboxPatch((x-0.5, y-0.3), 1.0, 0.6, 
                                       boxstyle="round,pad=0.1", 
                                       facecolor='#4ECDC4', ec='black', lw=2)
        ax.add_patch(rect)
        ax.text(x, y, nombre, ha='center', va='center', fontsize=12, fontweight='bold')

# Flechas
flechas = [
    ('a=2', '+'), ('b=3', '+'), ('+', 't=5'), 
    ('t=5', '×'), ('c=4', '×'), ('×', 'y=20')
]
for src, dst in flechas:
    x1, y1 = nodos[src]
    x2, y2 = nodos[dst]
    ax.annotate('', xy=(x2-0.4, y2), xytext=(x1+0.5, y1),
                arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 4.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Grafo de Computación: y = (a + b) × c', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Paso adelante (forward pass):")
print("  a=2, b=3 → temp = 2+3 = 5")
print("  temp=5, c=4 → y = 5×4 = 20")

In [ ]:
# =============================================================
# EJEMPLO 5: El paso hacia atrás (backpropagation) visual
# =============================================================

print("BACKPROPAGATION — Paso hacia atrás")
print("="*55)
print()
print("Expresión: y = (a + b) × c")
print("Con: a=2, b=3, c=4")
print()

a, b, c = 2.0, 3.0, 4.0
temp = a + b    # 5
y = temp * c    # 20

print("── FORWARD (adelante) ──")
print(f"  temp = a + b = {a} + {b} = {temp}")
print(f"  y = temp × c = {temp} × {c} = {y}")
print()

# Backpropagation: ¿cómo afecta cada variable a y?
print("── BACKWARD (atrás) ──")
print("  Empezamos desde y. dy/dy = 1 (siempre)")
print()

dy_dy = 1.0
print(f"  Nodo '×': y = temp × c")
dy_dtemp = c * dy_dy  # = 4
dy_dc = temp * dy_dy   # = 5
print(f"    dy/dtemp = c = {c}")
print(f"    dy/dc    = temp = {temp}")
print()

print(f"  Nodo '+': temp = a + b")
dy_da = 1.0 * dy_dtemp  # = 4
dy_db = 1.0 * dy_dtemp  # = 4
print(f"    dy/da = 1 × dy/dtemp = 1 × {dy_dtemp} = {dy_da}")
print(f"    dy/db = 1 × dy/dtemp = 1 × {dy_dtemp} = {dy_db}")
print()

print("── RESULTADO ──")
print(f"  dy/da = {dy_da}  (si a sube 1, y sube {dy_da})")
print(f"  dy/db = {dy_db}  (si b sube 1, y sube {dy_db})")
print(f"  dy/dc = {dy_dc}  (si c sube 1, y sube {dy_dc})")
print()

# Verificación
print("── VERIFICACIÓN ──")
h = 0.001
for nombre, val, otros in [('a', a, (b,c)), ('b', b, (a,c)), ('c', c, (a,b))]:
    if nombre == 'a':
        y1 = (val + h + otros[0]) * otros[1]
    elif nombre == 'b':
        y1 = (otros[0] + val + h) * otros[1]
    else:
        y1 = (otros[0] + otros[1]) * (val + h)
    deriv = (y1 - y) / h
    print(f"  dy/d{nombre} numérica = {deriv:.2f} ✓")

### 💡 Ideas clave hasta aquí

1. **Forward pass** = calcular el resultado de adelante hacia atrás
2. **Backward pass** = calcular derivadas de atrás hacia adelante
3. En el backward, usamos la **regla de la cadena** para propagar gradientes
4. Cada operación solo necesita saber su **derivada local**

| Operación | Derivada respecto a x | Derivada respecto a y |
|-----------|----------------------|----------------------|
| x + y | 1 | 1 |
| x × y | y | x |
| x² | 2x | — |
| ReLU(x) | 1 si x>0, sino 0 | — |

---
# Sección 4: Construyendo Nuestro Motor Autograd

## ¿Qué es "autograd"?

**Auto** + **grad** = Gradientes **automáticos**

En vez de calcular derivadas a mano, vamos a construir un sistema que:
1. **Registra** cada operación que hacemos (construye el grafo)
2. **Calcula** automáticamente todas las derivadas (backpropagation)

### Analogía: GPS con historial 🗺️

Imagina un GPS que:
- Registra cada giro que das (forward pass = grafo)
- Si te pierdes, puede decirte exactamente cómo regresar (backward pass = gradientes)

Vamos a construir esto en **menos de 50 líneas de código**.

In [ ]:
# =============================================================
# MOTOR AUTOGRAD — Versión educativa (simplificada)
# =============================================================

class Valor:
    """
    Un número que recuerda cómo fue creado.
    Puede calcular derivadas automáticamente.
    """
    
    def __init__(self, dato, _hijos=(), _operacion=''):
        self.dato = dato               # El número en sí
        self.grad = 0.0                 # La derivada (se llena en backward)
        self._hijos = set(_hijos)       # De dónde vino este valor
        self._operacion = _operacion    # Qué operación lo creó
        self._backward = lambda: None   # Función para calcular gradientes
    
    def __repr__(self):
        return f"Valor({self.dato:.4f}, grad={self.grad:.4f})"
    
    # --- Operación: SUMA ---
    def __add__(self, otro):
        otro = otro if isinstance(otro, Valor) else Valor(otro)
        resultado = Valor(self.dato + otro.dato, (self, otro), '+')
        
        def _backward():
            # Derivada de suma: ambos reciben el gradiente completo
            self.grad += resultado.grad
            otro.grad += resultado.grad
        
        resultado._backward = _backward
        return resultado
    
    # --- Operación: MULTIPLICACIÓN ---
    def __mul__(self, otro):
        otro = otro if isinstance(otro, Valor) else Valor(otro)
        resultado = Valor(self.dato * otro.dato, (self, otro), '×')
        
        def _backward():
            # Derivada de multiplicación: se cruzan los valores
            self.grad += otro.dato * resultado.grad
            otro.grad += self.dato * resultado.grad
        
        resultado._backward = _backward
        return resultado
    
    # --- Operación: POTENCIA ---
    def __pow__(self, n):
        resultado = Valor(self.dato ** n, (self,), f'**{n}')
        
        def _backward():
            self.grad += n * (self.dato ** (n - 1)) * resultado.grad
        
        resultado._backward = _backward
        return resultado
    
    # --- Operación: ReLU (activación) ---
    def relu(self):
        resultado = Valor(max(0, self.dato), (self,), 'ReLU')
        
        def _backward():
            # Si el valor era positivo, pasa el gradiente; si no, lo bloquea
            self.grad += (self.dato > 0) * resultado.grad
        
        resultado._backward = _backward
        return resultado
    
    # Operaciones auxiliares para que funcionen expresiones como 3 * valor
    def __radd__(self, otro): return self + otro
    def __rmul__(self, otro): return self * otro
    def __neg__(self): return self * -1
    def __sub__(self, otro): return self + (-otro)
    
    # --- BACKWARD: Calcula TODOS los gradientes automáticamente ---
    def backward(self):
        # Paso 1: Ordenar nodos (orden topológico)
        orden = []
        visitados = set()
        def construir_orden(v):
            if v not in visitados:
                visitados.add(v)
                for hijo in v._hijos:
                    construir_orden(hijo)
                orden.append(v)
        construir_orden(self)
        
        # Paso 2: Propagar gradientes de atrás hacia adelante
        self.grad = 1.0  # dself/dself = 1
        for nodo in reversed(orden):
            nodo._backward()

print("✓ Clase 'Valor' definida exitosamente")
print("  Soporta: suma, multiplicación, potencia, ReLU")
print("  Calcula gradientes automáticamente con .backward()")

In [ ]:
# =============================================================
# EJEMPLO 6: Probando nuestro autograd
# =============================================================

print("PRUEBA DEL MOTOR AUTOGRAD")
print("="*50)
print()

# Definimos variables
a = Valor(2.0)
b = Valor(3.0)
c = Valor(4.0)

# Hacemos operaciones (se construye el grafo automáticamente)
y = (a + b) * c

print(f"a = {a.dato}, b = {b.dato}, c = {c.dato}")
print(f"y = (a + b) × c = ({a.dato} + {b.dato}) × {c.dato} = {y.dato}")
print()

# Calculamos gradientes automáticamente
y.backward()

print("Gradientes calculados automáticamente:")
print(f"  dy/da = {a.grad}  (esperado: {c.dato})")
print(f"  dy/db = {b.grad}  (esperado: {c.dato})")
print(f"  dy/dc = {c.grad}  (esperado: {a.dato + b.dato})")
print()
print("¡Todos los gradientes son correctos! ✓")

In [ ]:
# =============================================================
# EJEMPLO 7: Un ejemplo más complejo
# =============================================================

print("EJEMPLO COMPLEJO: y = (x₁×w₁ + x₂×w₂)²")
print("="*50)
print()

# Simulamos entradas y pesos de una neurona
x1 = Valor(1.0)   # Entrada 1
w1 = Valor(0.5)   # Peso 1
x2 = Valor(2.0)   # Entrada 2
w2 = Valor(-0.3)  # Peso 2

# Forward pass
z = x1 * w1 + x2 * w2   # Combinación lineal
y = z ** 2                # Función de pérdida simple

print(f"x₁={x1.dato}, w₁={w1.dato}, x₂={x2.dato}, w₂={w2.dato}")
print(f"z = x₁×w₁ + x₂×w₂ = {x1.dato}×{w1.dato} + {x2.dato}×{w2.dato} = {z.dato}")
print(f"y = z² = {z.dato}² = {y.dato}")
print()

# Backward pass
y.backward()

print("Gradientes (automáticos):")
print(f"  dy/dw₁ = {w1.grad:.4f}")
print(f"  dy/dw₂ = {w2.grad:.4f}")
print()

# Verificación manual
# y = z², dy/dz = 2z = 2(-0.1) = -0.2
# z = x1*w1 + x2*w2
# dz/dw1 = x1 = 1.0, dz/dw2 = x2 = 2.0
# dy/dw1 = dy/dz * dz/dw1 = -0.2 * 1.0 = -0.2
# dy/dw2 = dy/dz * dz/dw2 = -0.2 * 2.0 = -0.4
print("Verificación manual (regla de la cadena):")
print(f"  dy/dz = 2z = 2×{z.dato} = {2*z.dato}")
print(f"  dy/dw₁ = dy/dz × dz/dw₁ = {2*z.dato} × {x1.dato} = {2*z.dato*x1.dato:.4f} ✓")
print(f"  dy/dw₂ = dy/dz × dz/dw₂ = {2*z.dato} × {x2.dato} = {2*z.dato*x2.dato:.4f} ✓")

---
# Sección 5: Una Neurona que Aprende

## ¿Qué es una neurona artificial?

Una neurona es una función muy simple:

```
salida = activación(peso₁ × entrada₁ + peso₂ × entrada₂ + ... + sesgo)
```

### Analogía: Un juez de competencia 🏆

Imagina un juez que califica presentaciones:
- Cada criterio (contenido, presentación, originalidad) tiene un **peso** (importancia)
- El juez **multiplica** cada nota por su peso y **suma** todo
- Si la suma total pasa un umbral, da una calificación positiva

**Aprender** = ajustar los pesos hasta que el juez califique correctamente

Vamos a hacer que una neurona aprenda algo simple: **la función y = 2x**

In [ ]:
# =============================================================
# EJEMPLO 8: Una neurona aprende y = 2x
# =============================================================
import random

print("UNA NEURONA APRENDE: y = 2x")
print("="*50)
print()

# Datos de entrenamiento
# Queremos que aprenda: cuando x=1 → y=2, x=2 → y=4, x=3 → y=6, ...
datos_x = [1.0, 2.0, 3.0, 4.0, 5.0]
datos_y = [2.0, 4.0, 6.0, 8.0, 10.0]  # y = 2x

# La neurona empieza con un peso aleatorio
random.seed(42)
w = Valor(random.uniform(-1, 1))  # Peso inicial aleatorio
tasa_aprendizaje = 0.01

print(f"Peso inicial: w = {w.dato:.4f}")
print(f"Objetivo: aprender que w debería ser 2.0")
print(f"Tasa de aprendizaje: {tasa_aprendizaje}")
print()

# Entrenamiento
historial_pesos = [w.dato]
historial_error = []

for epoca in range(20):
    # --- Forward pass: calcular predicciones y error ---
    error_total = Valor(0.0)
    for x, y_real in zip(datos_x, datos_y):
        prediccion = w * x           # y_pred = w * x
        error = (prediccion - y_real) ** 2  # Error cuadrático
        error_total = error_total + error
    
    # --- Backward pass: calcular gradiente ---
    error_total.backward()
    
    # Guardar historial
    historial_error.append(error_total.dato)
    
    # Mostrar progreso
    if epoca < 5 or epoca % 5 == 0:
        print(f"Época {epoca:2d}: w = {w.dato:7.4f}, error = {error_total.dato:8.4f}, grad = {w.grad:8.4f}")
    
    # --- Actualizar peso (descenso de gradiente) ---
    w = Valor(w.dato - tasa_aprendizaje * w.grad)
    historial_pesos.append(w.dato)

print()
print(f"Peso final: w = {w.dato:.4f}")
print(f"Peso esperado: 2.0000")
print(f"¡La neurona aprendió que y = {w.dato:.2f}x ≈ 2x! ✓")

In [ ]:
# =============================================================
# Visualización del aprendizaje
# =============================================================
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: Evolución del peso
ax1.plot(historial_pesos, 'b-o', markersize=4, linewidth=2)
ax1.axhline(y=2.0, color='r', linestyle='--', label='Objetivo (w=2.0)')
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Valor del peso (w)', fontsize=12)
ax1.set_title('El peso converge al valor correcto', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Gráfico 2: Evolución del error
ax2.plot(historial_error, 'r-o', markersize=4, linewidth=2)
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Error total', fontsize=12)
ax2.set_title('El error disminuye con cada época', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Izquierda: El peso w se acerca gradualmente a 2.0")
print("Derecha: El error se reduce en cada iteración")

### 🔑 ¿Qué acabamos de hacer?

1. Empezamos con un peso aleatorio
2. Calculamos el error (qué tan lejos estamos del resultado correcto)
3. Usamos `.backward()` para saber cómo ajustar el peso
4. Ajustamos el peso en la dirección correcta
5. Repetimos hasta que el error sea pequeño

**Esto es exactamente lo que hace PyTorch, TensorFlow y tinygrad por debajo.**

---
# Sección 6: Red Neuronal Completa (MLP)

Ahora construiremos una red con **múltiples neuronas** organizadas en capas.

```
Entrada      Capa oculta      Salida
  x₁ ──┬──── n₁ ──┐
        ├──── n₂ ──┼──── resultado
  x₂ ──┴──── n₃ ──┘
```

Cada neurona tiene sus propios pesos que se ajustan durante el entrenamiento.

In [ ]:
# =============================================================
# Red Neuronal desde cero usando nuestra clase Valor
# =============================================================
import random

class Neurona:
    """Una neurona: pesos × entradas + sesgo, luego ReLU."""
    def __init__(self, n_entradas):
        # Inicializar pesos aleatorios
        self.pesos = [Valor(random.uniform(-1, 1)) for _ in range(n_entradas)]
        self.sesgo = Valor(0.0)
    
    def __call__(self, x):
        # peso₁×x₁ + peso₂×x₂ + ... + sesgo
        suma = sum((pi * xi for pi, xi in zip(self.pesos, x)), self.sesgo)
        return suma.relu()
    
    def parametros(self):
        return self.pesos + [self.sesgo]


class Capa:
    """Una capa de múltiples neuronas."""
    def __init__(self, n_entradas, n_salidas):
        self.neuronas = [Neurona(n_entradas) for _ in range(n_salidas)]
    
    def __call__(self, x):
        salidas = [n(x) for n in self.neuronas]
        return salidas[0] if len(salidas) == 1 else salidas
    
    def parametros(self):
        return [p for n in self.neuronas for p in n.parametros()]


class MLP:
    """Perceptrón Multicapa: varias capas conectadas."""
    def __init__(self, n_entradas, tamanos_capas):
        tamanos = [n_entradas] + tamanos_capas
        self.capas = [Capa(tamanos[i], tamanos[i+1]) for i in range(len(tamanos_capas))]
    
    def __call__(self, x):
        for capa in self.capas:
            x = capa(x)
        return x
    
    def parametros(self):
        return [p for capa in self.capas for p in capa.parametros()]

print("✓ Clases Neurona, Capa y MLP definidas")
print("  Neurona: pesos × entradas + sesgo → ReLU")
print("  Capa: varias neuronas en paralelo")
print("  MLP: varias capas en serie")

In [ ]:
# =============================================================
# EJEMPLO 9: Entrenando una red para clasificar
# =============================================================
random.seed(42)

# Datos: ¿el punto está arriba o abajo de la línea y=x?
#   Entrada: (x₁, x₂)  →  Salida: +1 (arriba) o -1 (abajo)
datos = [
    ([2.0, 3.0], 1.0),    # (2,3): 3 > 2, arriba → +1
    ([3.0, 1.0], -1.0),   # (3,1): 1 < 3, abajo → -1
    ([1.0, 4.0], 1.0),    # (1,4): 4 > 1, arriba → +1
    ([4.0, 2.0], -1.0),   # (4,2): 2 < 4, abajo → -1
    ([1.0, 1.5], 1.0),    # (1,1.5): 1.5 > 1, arriba → +1
    ([2.0, 0.5], -1.0),   # (2,0.5): 0.5 < 2, abajo → -1
]

print("ENTRENANDO RED NEURONAL")
print("="*50)
print()
print("Tarea: clasificar si un punto (x₁,x₂) está")
print("       arriba (+1) o abajo (-1) de la línea y=x")
print()

# Crear red: 2 entradas → 4 neuronas ocultas → 1 salida
red = MLP(2, [4, 1])
n_params = len(red.parametros())
print(f"Red creada: 2 → 4 → 1")
print(f"Total de parámetros: {n_params}")
print()

# Entrenamiento
tasa = 0.05
historial_loss = []

for epoca in range(30):
    # Forward: hacer predicciones
    perdida_total = Valor(0.0)
    aciertos = 0
    
    for x, y_real in datos:
        prediccion = red(x)
        # Pérdida: queremos que predicción se acerque a y_real
        perdida = (prediccion - y_real) ** 2
        perdida_total = perdida_total + perdida
        
        # Contar aciertos (mismo signo = acierto)
        if (prediccion.dato > 0) == (y_real > 0):
            aciertos += 1
    
    historial_loss.append(perdida_total.dato)
    
    # Mostrar progreso
    if epoca < 3 or epoca % 5 == 0 or epoca == 29:
        precision = aciertos / len(datos) * 100
        print(f"Época {epoca:2d}: pérdida = {perdida_total.dato:8.4f}, precisión = {precision:.0f}% ({aciertos}/{len(datos)})")
    
    # Backward: calcular gradientes
    # Primero limpiar gradientes anteriores
    for p in red.parametros():
        p.grad = 0.0
    perdida_total.backward()
    
    # Actualizar pesos
    for p in red.parametros():
        p.dato -= tasa * p.grad

print()
print("Predicciones finales:")
for x, y_real in datos:
    pred = red(x)
    signo = "+" if pred.dato > 0 else "-"
    correcto = "✓" if (pred.dato > 0) == (y_real > 0) else "✗"
    print(f"  ({x[0]}, {x[1]}) → pred: {signo}{abs(pred.dato):.2f}, real: {y_real:+.0f} {correcto}")

In [ ]:
# Visualización del entrenamiento
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: Pérdida
ax1.plot(historial_loss, 'r-', linewidth=2)
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Pérdida', fontsize=12)
ax1.set_title('La pérdida baja durante el entrenamiento', fontsize=13)
ax1.grid(True, alpha=0.3)

# Gráfico 2: Los datos y la frontera de decisión
for x, y_real in datos:
    color = 'blue' if y_real > 0 else 'red'
    marker = '^' if y_real > 0 else 'v'
    label = 'Arriba (+1)' if y_real > 0 and color == 'blue' else ('Abajo (-1)' if y_real < 0 and color == 'red' else '')
    ax2.scatter(x[0], x[1], c=color, marker=marker, s=100, edgecolors='black', linewidth=1.5)

# Línea y=x de referencia
ax2.plot([0, 5], [0, 5], 'g--', linewidth=2, label='y = x (frontera)')
ax2.set_xlabel('x₁', fontsize=12)
ax2.set_ylabel('x₂', fontsize=12)
ax2.set_title('Datos: ▲azul=arriba, ▼rojo=abajo', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 5)
ax2.set_ylim(0, 5)

plt.tight_layout()
plt.show()

### 🔑 Resumen: El ciclo de entrenamiento

Cada vez que entrenamos, hacemos estos 4 pasos:

```
┌──────────────────────────────────────────┐
│  1. FORWARD    →  Calcular predicciones  │
│  2. PÉRDIDA    →  Medir el error         │
│  3. BACKWARD   →  Calcular gradientes    │
│  4. ACTUALIZAR →  Ajustar pesos          │
│         ↓                                │
│    Repetir hasta que el error sea bajo    │
└──────────────────────────────────────────┘
```

Nuestro motor autograd se encarga del paso 3 automáticamente.

---
# Sección 7: Comparación con tinygrad Real

Nuestro motor autograd funciona, pero es muy básico:
- Solo maneja **números individuales** (escalares)
- Es **lento** para datos grandes

## ¿Cómo lo hace tinygrad?

tinygrad (~19,000 líneas de código) agrega:

| Característica | Nuestro motor | tinygrad |
|---------------|---------------|----------|
| Tipo de dato | Escalares | Tensores (matrices) |
| Velocidad | Lento (Python puro) | Rápido (GPU) |
| Evaluación | Inmediata | Perezosa (lazy) |
| Optimización | Ninguna | Fusión de kernels |
| Hardware | Solo CPU | CPU, GPU, Metal, etc. |

### Evaluación perezosa (lazy evaluation)

Nuestro motor calcula cada operación inmediatamente.  
tinygrad **espera** y acumula operaciones, luego las ejecuta todas juntas de forma optimizada.

**Analogía**: Es como ir al supermercado:
- **Eager (nuestro motor)**: Vas al super cada vez que necesitas un ingrediente
- **Lazy (tinygrad)**: Haces una lista y vas una sola vez

In [ ]:
# =============================================================
# EJEMPLO 10: Simulando evaluación perezosa
# =============================================================

class TensorPerezoso:
    """Simula cómo tinygrad acumula operaciones antes de ejecutar."""
    
    def __init__(self, nombre, dato=None):
        self.nombre = nombre
        self.dato = dato
        self.operaciones = []  # Lista de operaciones pendientes
    
    def __add__(self, otro):
        resultado = TensorPerezoso(f"({self.nombre} + {otro.nombre})")
        resultado.operaciones = self.operaciones + otro.operaciones + [f"{self.nombre} + {otro.nombre}"]
        return resultado
    
    def __mul__(self, otro):
        resultado = TensorPerezoso(f"({self.nombre} × {otro.nombre})")
        resultado.operaciones = self.operaciones + otro.operaciones + [f"{self.nombre} × {otro.nombre}"]
        return resultado
    
    def realize(self):
        """¡Ahora sí ejecutamos todo!"""
        print(f"Ejecutando {len(self.operaciones)} operaciones acumuladas:")
        for i, op in enumerate(self.operaciones, 1):
            print(f"  {i}. {op}")

print("EVALUACIÓN PEREZOSA (LAZY)")
print("="*50)
print()

a = TensorPerezoso("a")
b = TensorPerezoso("b")
c = TensorPerezoso("c")

# Estas operaciones NO se ejecutan todavía
print("Definiendo operaciones (no se ejecutan aún)...")
d = a + b
e = d * c
f = e + a
print(f"Expresión acumulada: {f.nombre}")
print()

# Ahora sí ejecutamos todo de golpe
print("Llamando .realize() — ¡ahora sí se ejecuta!")
f.realize()
print()
print("Ventaja: tinygrad puede optimizar y fusionar")
print("estas operaciones antes de ejecutarlas.")

In [ ]:
# =============================================================
# EJEMPLO 11: Comparación visual de frameworks
# =============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 6))

# Datos
frameworks = ['Nuestro motor\n(este notebook)', 'micrograd\n(Karpathy)', 'tinygrad\n(Hotz)', 'PyTorch\n(Meta)']
lineas = [50, 100, 19000, 3000000]
colores = ['#4ECDC4', '#45B7D1', '#FF6B6B', '#96CEB4']
log_lineas = [1.7, 2, 4.3, 6.5]  # escala visual

bars = ax.barh(frameworks, log_lineas, color=colores, edgecolor='black', linewidth=1.5, height=0.6)

# Etiquetas con líneas reales
for bar, linea in zip(bars, lineas):
    width = bar.get_width()
    label = f'{linea:,} líneas'
    ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, label,
            va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Complejidad (escala logarítmica)', fontsize=12)
ax.set_title('Comparación de frameworks de autograd', fontsize=14, fontweight='bold')
ax.set_xlim(0, 8)
ax.set_xticks([])
plt.tight_layout()
plt.show()

print("Todos usan el MISMO principio que aprendimos:")
print("grafo de computación + regla de la cadena + backpropagation")
print()
print("La diferencia está en optimizaciones, hardware y escala.")

---
# Resumen Final

## Lo que aprendimos hoy:

| # | Concepto | En una frase |
|---|---------|-------------|
| 1 | **Derivada** | Mide cuánto cambia la salida si cambio un poquito la entrada |
| 2 | **Regla de la cadena** | Los efectos se multiplican a lo largo de una cadena de operaciones |
| 3 | **Grafo de computación** | Diagrama que registra todas las operaciones realizadas |
| 4 | **Autograd** | Sistema que calcula derivadas automáticamente usando el grafo |
| 5 | **Backpropagation** | Propagar gradientes de atrás hacia adelante en el grafo |
| 6 | **Descenso de gradiente** | Ajustar pesos en la dirección que reduce el error |
| 7 | **Evaluación perezosa** | Acumular operaciones y ejecutarlas juntas (optimizado) |

## El ciclo completo:

```
Datos → Forward Pass → Pérdida → Backward Pass → Actualizar Pesos → Repetir
         (grafo)       (error)     (autograd)      (gradiente)
```

## ¿Qué sigue?

- Explorar el notebook técnico completo: `Tinygrad_Autograd_Grafos_Computacion.ipynb`
- Instalar tinygrad: `pip install tinygrad`
- Probar con tensores reales y GPU
- Implementar redes más grandes (CNNs, transformers)

---
*Universidad Privada del Norte — Escuela de Posgrado*  
*Bloque B: Ingeniería y Frameworks — Tinygrad*